In [ ]:

import pandas as pd
from pathlib import Path

# Βρίσκουμε το CSV μέσα στον φάκελο data
csv_files = list(Path("data").glob("*.csv"))

print("CSV files:", [file.name for file in csv_files])

# Προς το παρόν πρέπει να υπάρχει μόνο το cleaned CSV
assert len(csv_files) == 1, "cleaned_ids2018_sampled"

csv_path = csv_files[0]

# Διαβάζουμε μόνο τις πρώτες 1000 γραμμές
df = pd.read_csv(csv_path, nrows=1000)

# Αφαιρούμε τυχόν κενά από τα ονόματα των στηλών
df.columns = df.columns.str.strip()

print("CSV size (MB):", round(csv_path.stat().st_size / 1024**2, 2))
print("Sample shape:", df.shape)

# Εμφανίζουμε τις πρώτες 5 γραμμές
display(df.head())

# Εμφανίζουμε τις στήλες και τους τύπους δεδομένων
df.info()

# Μετράμε τις ελλιπείς τιμές στο δείγμα
print("Missing values in sample:", df.isna().sum().sum())


In [ ]:

# Μετράμε τις κατηγορίες σε ολόκληρο το dataset
# χωρίς να φορτώσουμε όλες τις στήλες στη RAM.

label_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(
    csv_path,
    usecols=["Label"],
    chunksize=50000
):
    counts = chunk["Label"].value_counts(dropna=False)
    label_counts = label_counts.add(
        counts, fill_value=0
    )

label_counts = label_counts.astype("int64")
total_rows = int(label_counts.sum())

print("Total rows:", total_rows)

print("\nLabel distribution:")
print(label_counts.sort_index())

print("\nPercentage per label:")
print((label_counts / total_rows * 100).round(2))


In [ ]:

import numpy as np

# Μετρητές για ελλιπείς και άπειρες τιμές
missing_total = pd.Series(0, index=df.columns, dtype="int64")
inf_total = pd.Series(0, index=df.columns, dtype="int64")

# Διαβάζουμε το CSV σε τμήματα των 20.000 γραμμών
for chunk in pd.read_csv(csv_path, chunksize=20000):

    chunk.columns = chunk.columns.str.strip()

    # Ελέγχουμε ελλιπείς τιμές (NaN)
    missing_total += chunk.isna().sum()

    # Ελέγχουμε άπειρες τιμές (inf, -inf)
    numeric = chunk.select_dtypes(include=[np.number])

    inf_counts = pd.Series(
        np.isinf(numeric.to_numpy(dtype="float64")).sum(axis=0),
        index=numeric.columns
    )

    inf_total += inf_counts

print("Total missing values:", missing_total.sum())
print("Total infinite values:", inf_total.sum())

print("\nColumns with missing values:")
print(missing_total[missing_total > 0])

print("\nColumns with infinite values:")
print(inf_total[inf_total > 0])


In [ ]:
import pandas as pd
import numpy as np
import joblib
import gc
from pathlib import Path
from sklearn.model_selection import train_test_split

folder = Path("processed")
folder.mkdir(parents=True, exist_ok=True)
source_path = csv_path

time_columns = [
    "Flow Duration",
    "Flow IAT Min", "Flow IAT Max",
    "Fwd IAT Min", "Fwd IAT Max",
    "Bwd IAT Min", "Bwd IAT Max",
]
min_max_pairs = [
    ("Flow IAT Min", "Flow IAT Max"),
    ("Fwd IAT Min", "Fwd IAT Max"),
    ("Bwd IAT Min", "Bwd IAT Max"),
    ("Idle Min", "Idle Max"),
]

def prepare_chunk(chunk):
    """Επιστρέφει έγκυρες γραμμές και αποτυπώματα όλων των features."""
    chunk.columns = chunk.columns.str.strip()
    invalid = (chunk[time_columns] < 0).any(axis=1)
    for min_col, max_col in min_max_pairs:
        invalid |= chunk[min_col] > chunk[max_col]
    valid = chunk.loc[~invalid].copy()
    network_features = valid.drop(columns=["Unnamed: 0", "Label"])
    signatures = pd.util.hash_pandas_object(
        network_features, index=False
    ).to_numpy()
    return valid, signatures, int(invalid.sum())

# Ορισμός ομάδων με πρώτο Label μόνο για stratification
first_group_labels = {}
invalid_source_rows = 0
valid_source_rows = 0

for chunk in pd.read_csv(source_path, chunksize=20000):
    valid, signatures, invalid_count = prepare_chunk(chunk)
    invalid_source_rows += invalid_count
    valid_source_rows += len(valid)
    for signature, label in zip(signatures, valid["Label"].to_numpy()):
        first_group_labels.setdefault(int(signature), int(label))

groups = list(first_group_labels)
strata = [first_group_labels[g] for g in groups]
train_groups, test_groups = train_test_split(
    groups, test_size=0.20, random_state=42, stratify=strata
)
train_groups, test_groups = set(train_groups), set(test_groups)
assert train_groups.isdisjoint(test_groups)

joblib.dump(
    {"train": train_groups, "test": test_groups},
    folder / "group_split.joblib"
)

print("Invalid source rows:", invalid_source_rows)
print("Valid source rows:", valid_source_rows)
print("Unique feature groups:", len(groups))
print("Duplicate feature rows:", valid_source_rows - len(groups))
print("Training groups:", len(train_groups))
print("Testing groups:", len(test_groups))
print("Overlapping groups:", len(train_groups & test_groups))

del first_group_labels, groups, strata, train_groups, test_groups
gc.collect()


In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

folder = Path("processed")
source_path = csv_path
split_info = joblib.load(folder / "group_split.joblib")
train_groups, test_groups = split_info["train"], split_info["test"]

# Διατηρούμε όλα τα αρχικά features στο grouped dataset.
# Τα σταθερά features θα εντοπιστούν ΜΟΝΟ στο training set στο επόμενο βήμα.
raw_columns = pd.read_csv(source_path, nrows=0).columns.str.strip().tolist()
output_columns = [col for col in raw_columns if col != "Unnamed: 0"]
assert "Label" in output_columns

# Πέρασμα 1: εντοπίζουμε ομάδες με αντιφατικά Labels
first_labels = {}
conflicting_groups = set()
for chunk in pd.read_csv(source_path, chunksize=20000):
    valid, signatures, _ = prepare_chunk(chunk)
    for signature, label in zip(signatures, valid["Label"].to_numpy()):
        h, label = int(signature), int(label)
        if h not in first_labels:
            first_labels[h] = label
        elif first_labels[h] != label:
            conflicting_groups.add(h)
print("Conflicting groups:", len(conflicting_groups))
del first_labels

# Πέρασμα 2: μία εγγραφή ανά μη αμφίσημη ομάδα στο σωστό split
train_path = folder / "train_grouped.csv"
test_path = folder / "test_grouped.csv"
for path in (train_path, test_path):
    if path.exists():
        path.unlink()

seen_groups = set()
train_count = 0
test_count = 0

for chunk in pd.read_csv(source_path, chunksize=20000):
    valid, signatures, _ = prepare_chunk(chunk)
    destinations = []
    for signature in signatures:
        h = int(signature)
        if h in conflicting_groups or h in seen_groups:
            destinations.append("skip")
            continue
        seen_groups.add(h)
        if h in train_groups:
            destinations.append("train")
        elif h in test_groups:
            destinations.append("test")
        else:
            raise ValueError("Unknown feature group")

    destinations = np.asarray(destinations)
    train_chunk = valid.loc[destinations == "train", output_columns]
    test_chunk = valid.loc[destinations == "test", output_columns]
    for output_path, frame in ((train_path, train_chunk), (test_path, test_chunk)):
        if not frame.empty:
            frame.to_csv(
                output_path, mode="a",
                header=not output_path.exists(), index=False
            )
    train_count += len(train_chunk)
    test_count += len(test_chunk)

expected_total = len(train_groups) + len(test_groups) - len(conflicting_groups)
assert train_count + test_count == expected_total
assert train_groups.isdisjoint(test_groups)
print("Training rows:", train_count)
print("Testing rows:", test_count)
print("Total rows:", train_count + test_count)
print("Expected rows:", expected_total)
print("Overlapping groups:", len(train_groups & test_groups))
print("Grouped datasets saved.")

del seen_groups, train_groups, test_groups


In [ ]:
import pandas as pd
import numpy as np
import json
import gc
from pathlib import Path

folder = Path("processed")
threshold = 0.95
chunk_size = 20000

# ΜΟΝΟ training δεδομένα για αποφάσεις επιλογής χαρακτηριστικών
X_train = pd.read_csv(
    folder / "train_grouped.csv",
    usecols=lambda col: col not in ("Label", "Unnamed: 0")
)

constant_features = X_train.columns[
    X_train.min(axis=0) == X_train.max(axis=0)
].tolist()
active_features = [c for c in X_train.columns if c not in constant_features]

print("Features before removing constants:", X_train.shape[1])
print("Constant features removed:", len(constant_features))
print("Features entering correlation analysis:", len(active_features))

corr_final = X_train[active_features].corr(method="pearson")
del X_train
gc.collect()

upper_final = corr_final.abs().where(
    np.triu(np.ones(corr_final.shape, dtype=bool), k=1)
)
high_pairs = int((upper_final > threshold).sum().sum())
removed_features = [
    col for col in upper_final.columns
    if (upper_final[col] > threshold).any()
]
selected_features = [
    col for col in corr_final.columns if col not in removed_features
]

selected_upper = corr_final.loc[selected_features, selected_features].abs().where(
    np.triu(np.ones((len(selected_features), len(selected_features)), dtype=bool), k=1)
)
remaining_pairs = int((selected_upper > threshold).sum().sum())
assert remaining_pairs == 0

print("High-correlation pairs:", high_pairs)
print("Correlated features removed:", len(removed_features))
print("Selected features:", len(selected_features))
print("Remaining high-correlation pairs:", remaining_pairs)

with open(folder / "feature_selection_final.json", "w", encoding="utf-8") as f:
    json.dump({
        "method": "Pearson correlation",
        "threshold": threshold,
        "constant_features": constant_features,
        "removed_features": removed_features,
        "selected_features": selected_features,
    }, f, indent=2)

columns_to_keep = selected_features + ["Label"]
for split in ("train", "test"):
    input_path = folder / f"{split}_grouped.csv"
    output_path = folder / f"{split}_final_selected.csv"
    total = 0
    for i, chunk in enumerate(pd.read_csv(
        input_path, usecols=columns_to_keep, chunksize=chunk_size
    )):
        chunk[columns_to_keep].to_csv(
            output_path,
            mode="w" if i == 0 else "a",
            header=(i == 0), index=False
        )
        total += len(chunk)
    print(f"{split.upper()} selected rows: {total}")


In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

folder = Path("processed")
chunk_size = 20000
scaler_final = MinMaxScaler()

for chunk in pd.read_csv(folder / "train_final_selected.csv", chunksize=chunk_size):
    scaler_final.partial_fit(chunk[selected_features])

joblib.dump(
    {"scaler": scaler_final, "features": selected_features},
    folder / "minmax_scaler_final.joblib"
)
print("Scaler fitted using training data only.")

for split in ("train", "test"):
    input_path = folder / f"{split}_final_selected.csv"
    output_path = folder / f"{split}_final_scaled.csv"
    total = 0
    outside_range = 0
    for i, chunk in enumerate(pd.read_csv(input_path, chunksize=chunk_size)):
        values = scaler_final.transform(chunk[selected_features]).astype(np.float32)
        assert np.isfinite(values).all(), f"Non-finite values in {split}"
        outside_range += int(((values < -1e-5) | (values > 1 + 1e-5)).sum())
        scaled = pd.DataFrame(values, columns=selected_features)
        scaled["Label"] = chunk["Label"].to_numpy()
        scaled.to_csv(
            output_path, mode="w" if i == 0 else "a",
            header=(i == 0), index=False
        )
        total += len(scaled)
    print(f"{split.upper()}: rows={total}, features={len(selected_features)}, "
          f"out-of-range values={outside_range}, saved={output_path}")
    if split == "train":
        assert outside_range == 0

print("Final preprocessing completed.")


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

folder = Path("processed")
with open(folder / "feature_selection_final.json", encoding="utf-8") as f:
    feature_info = json.load(f)
features = feature_info["selected_features"]

for split in ("train", "test"):
    selected = pd.read_csv(folder / f"{split}_final_selected.csv")
    scaled = pd.read_csv(folder / f"{split}_final_scaled.csv")
    assert len(selected) == len(scaled)
    assert list(scaled.columns) == features + ["Label"]
    assert np.array_equal(selected["Label"].to_numpy(), scaled["Label"].to_numpy())
    assert np.isfinite(scaled[features].to_numpy()).all()
    print(f"{split.upper()}: {len(scaled)} rows, {len(features)} features, "
          f"label unchanged; finite values confirmed")

print("Ready for ML experiments, subject to confirming the numerical Label mapping.")
